In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import tempfile
import zipfile
import os
import rasterio
import numpy as np
import pandas as pd
from rasterio.mask import mask
import ocha_stratus as stratus

# --------------------------------------------------
# 0. Robust session (USGS drops connections)
# --------------------------------------------------
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504],
)
session.mount("https://", HTTPAdapter(max_retries=retries))


def download_zip(url, out_path):
    with session.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


# --------------------------------------------------
# 1. List matching ZIP files
# --------------------------------------------------
BASE_URL = "https://edcftp.cr.usgs.gov/project/fews/africa/east/dekadal/wrsi-chirps-etos/eastk/"
resp = session.get(BASE_URL)
soup = BeautifulSoup(resp.text, "html.parser")

zips = []
for a in soup.find_all("a"):
    href = a.get("href")
    if href and href.lower().endswith(("09ek.zip", "12ek.zip")):
        zips.append(BASE_URL + href)

print(f"Found {len(zips)} matching ZIP files.")

# --------------------------------------------------
# 2. Load ADM2 boundaries
# --------------------------------------------------
adm2 = stratus.codab.load_codab_from_fieldmaps("ETH", admin_level=2)
adm2 = adm2.to_crs(epsg=4326)

results = []

# --------------------------------------------------
# 3. Process each ZIP
# --------------------------------------------------
for zip_url in zips:
    fname = os.path.basename(zip_url).lower()
    month = "Mar" if "09ek" in fname else "Apr" if "12ek" in fname else None

    print(f"Processing {fname} → {month}")

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".zip")
    tmp.close()

    try:
        download_zip(zip_url, tmp.name)

        # sanity check
        with zipfile.ZipFile(tmp.name, "r") as z:
            z.testzip()
            tif_files = [f for f in z.namelist() if f.lower().endswith("do.tif")]

            if not tif_files:
                print("  → no do.tif found, skipping")
                continue

            tif_name = tif_files[0]
            z.extract(tif_name, path=os.path.dirname(tmp.name))
            tif_path = os.path.join(os.path.dirname(tmp.name), tif_name)

        # --------------------------------------------------
        # 4. Zonal stats
        # --------------------------------------------------
        with rasterio.open(tif_path) as src:
            adm2_proj = adm2.to_crs(src.crs) 
            for _, row in adm2.iterrows():
                try:
                    out_img, _ = mask(src, [row.geometry], crop=True)
                except ValueError:
                    continue

                data = out_img[0]
                valid = data[data != src.nodata]

                mean_val = np.nan
                if valid.size:
                    mean_val = float(valid.mean())

                results.append(
                    {
                        "pcode": row["adm2_pcode"],
                        "adm2_name": row["adm2_name"],
                        "month": month,
                        "mean_val": mean_val,
                    }
                )

        os.remove(tif_path)

    except Exception as e:
        print("  → failed:", e)

    finally:
        if os.path.exists(tmp.name):
            os.remove(tmp.name)

# --------------------------------------------------
# 5. Build DF and count
# --------------------------------------------------
df = pd.DataFrame(results)
df["mean_val"] = pd.to_numeric(df["mean_val"], errors="coerce")

count_below_60 = (
    df[df["mean_val"] < 60]
    .groupby("month")["pcode"]
    .nunique()
    .reset_index(name="zones_below_60")
)

print(count_below_60)
